# 参数辨识

## 1. 生成轨迹, 采集数据, 如果已经采集了数据则可以跳过
- 生成激励轨迹,并仿真验证, 真机采集, 如已采集数据可跳过

In [ ]:
# 运行参数
traj = 'traj1'
generate_new_traj = False   # 生成新轨迹并覆盖旧的同名文件
sim_check = False
real_capture = False
plot_records = False
is_anth = False
is_left = False

# 轨迹生成参数
import numpy as np
import sympy, sympybotics, pickle, os, sys
from pathlib import Path
from support_funcs.regr_data_proc import read_data, diff_and_filt_data, gen_regr_matrices
import matplotlib.pyplot as plt
import math

wf = 0.12 * np.pi # 轨迹基频
L = 3          # 轨迹谐波数
duration = 60.0 # 轨迹时长(s)
dt = 0.01
ndof = 7

# 初始化参数和路径
project_root = Path.cwd()
sys.path.insert(0, str(project_root))
robot_type = 'w1_anth' if is_anth else 'w1_indst'
arm_type = 'left' if is_left else 'right'
traj_dir = project_root / 'data' / 'trajectories'
rec_dir = project_root / 'data' / 'recdata' / robot_type / arm_type
cad_dir = project_root / 'data' / 'robotparams' / robot_type / arm_type
model_dir = project_root / 'tmp' / 'robotmodels' / robot_type
proc_dir = project_root / 'tmp' / 'procdata' / robot_type
sdpa_dir = project_root / 'tmp' / 'sdpa_dat' / robot_type
solution_dir = project_root / 'tmp' / 'solutions' / robot_type
for d in (traj_dir, rec_dir, cad_dir, model_dir, proc_dir, sdpa_dir, solution_dir):
    d.mkdir(parents=True, exist_ok=True)

joint_limits = [
    (-2.9670597283903604, 2.9670597283903604),  # J1
    (-2.0943951023931953, 1.5707963267948966),  # J2
    (-2.9670597283903604, 2.9670597283903604),  # J3
    (-2.356194490192345, 1.5707963267948966),  # J4
    (-2.9670597283903604, 2.96705972839036041),  # J5
    (-0.7853981633974483, 0.7853981633974483),  # J6
    (-1.5707963267948966, 1.0471975511965976)   # J7
]

traj_path = os.path.join(traj_dir, f"{traj}.dat")
if generate_new_traj:
    from data.trajectory_plot import *
    from data.trajectory_gen import *
    try:
        rclpy.shutdown()
    except:
        pass
    t, q = generate_q_with_limits(joint_limits, wf=wf, L=L, duration=duration, dt=dt) # 生成轨迹
    save_q(q, out_dir=traj_path) # 保存
    trajectory_plot(str(traj_path), ndof, joint_limits)

# 在仿真中测试轨迹的可行性，先启动仿真节点run_sim.sh（适用于W1）
if sim_check:
    from data.trajectory_exe import *
    try:
        rclpy.shutdown()
    except:
        pass
    exe_in_sim(is_left, str(traj_path), 1/dt)

# 运行真实轨迹，保证执行轨迹和状态反馈的频率相同, 记录状态 (t, q, tau)
if real_capture:
    from data.trajectory_exe import *
    try:
        rclpy.shutdown()
    except:
        pass
    rec_path = os.path.join(rec_dir, f"rbtlog_{traj}.dat")
    exe_in_real(is_left, traj_path, rec_path, 1/dt)

if plot_records or real_capture:
    # 读取数据并可视化 Read Data and Visualize
    rec_path = os.path.join(rec_dir, f"rbtlog_{traj}.dat")
    traj_path = os.path.join(traj_dir, f"{traj}.dat")
    t_raw, q_raw, tau_raw, t_ref, q_ref = read_data(ndof, dt, rec_path, traj_path)
    n_joints = q_raw.shape[1]
    cols = 3
    rows = math.ceil(n_joints / cols)

    # 统一取样长度，避免不同时间数组长度不一致
    len_tref = len(t_ref)
    len_traw = len(t_raw)
    len_ttau = tau_raw.shape[0]
    min_len = int(min(len_tref, len_traw, len_ttau))

    # 截断并对齐时间与信号
    t_ref_cut = np.asarray(t_ref[:min_len])
    t_raw_cut = np.asarray(t_raw[:min_len])
    q_ref_cut = np.asarray(q_ref[:min_len, :])
    q_raw_cut = np.asarray(q_raw[:min_len, :])
    tau_cut = np.asarray(tau_raw[:min_len, :])

    fig = plt.figure(figsize=(4 * cols, 2.5 * rows))

    for j in range(n_joints):
        ax1 = fig.add_subplot(rows, cols, j + 1)
        l_ref, = ax1.plot(t_ref_cut, q_ref_cut[:, j] if is_left else -q_ref_cut[:, j], label='Reference q', color='tab:blue', linewidth=1)
        l_meas, = ax1.plot(t_raw_cut, q_raw_cut[:, j], label='Measured q', color='tab:green', alpha=0.8, linewidth=1)
        ax1.set_title(f'Joint {j+1}')
        ax1.grid(True, linestyle='--', alpha=0.5)
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Angle (rad)', color='tab:blue')
        ax1.tick_params(axis='y', labelcolor='tab:blue')
        ax2 = ax1.twinx()
        l_tau, = ax2.plot(t_raw_cut, tau_cut[:, j], label='Measured τ (Nm)', color='tab:red', linewidth=0.5, linestyle='--')
        ax2.set_ylabel('Torque (Nm)', color='tab:red')
        ax2.tick_params(axis='y', labelcolor='tab:red')
        lines = [l_ref, l_meas, l_tau]
        labels = [ln.get_label() for ln in lines]
        ax1.legend(lines, labels, loc='upper right', fontsize='small')
    plt.tight_layout()
    plt.show()
    del q_raw, tau_raw, t_raw, t_ref, q_ref

## 2. 动力学参数计算
- 基本参数输入：包括DH参数，关节摩擦力模型，重力方向等
- sympybotics生成机器人符号公式
- 保存生成的机器人对象，便于直接加载
- 生成基本动力学参数, 待辨识的最小参数集

### 2.1 LMI矩阵定义 LMI Matrix Definitions

- 构造每个连杆的“空间惯性”LMI块：

$$
D_i = \begin{bmatrix}
L & S(l)^T \\
S(l) & m I_3
\end{bmatrix}
$$

其中：

- $L$：3x3 转动惯量张量（关于参考点，线性于参数）  
- $l$：质量一阶矩 $= m\cdot c$（$c$ 为质心坐标），$S(l)$ 为其反对称矩阵  
- $m$：质量

物理可行性约束：$D_i \succeq 0$，用于后续 LMI/SDP 中保证惯性参数物理合理。

In [ ]:
from lmi_sdp import init_lmi_latex_printing
from support_funcs.utils import skew
from sympy import BlockMatrix, Matrix, eye, Identity

init_lmi_latex_printing()

# 构建机器人模型
rbtdef = sympybotics.RobotDef("W1 Arm",
            [("-pi/2", 0, 0.1025, "q"),
             ("pi/2",  0, 0.0,    "q"),
             ("-pi/2", 0, 0.26,   "q"),
             ("pi/2",  0, 0.0,    "q"),
             ("-pi/2", 0, 0.264,  "q"),
             ("pi/2",  0, 0.0,    "q"),
             (0,       0, 0.0,    "q")],
            dh_convention="standard")
rbtdef.frictionmodel = {'Coulomb', 'viscous', 'offset'}
rbtdef.driveinertiamodel = 'simplified'
if is_left:
    rbtdef.gravityacc = sympy.Matrix([-9.81, 0.0, 0.0])
    if is_anth:
        joint_offset = np.array([0.0, 0.0, -np.pi/2.0, 0.0, 0.0, -np.pi/2.0, 0.0])
    else:
        joint_offset = np.array([0.0, 0.0, -np.pi/2.0, 0.0, 0.0, 0.0, 0.0])
else:
    rbtdef.gravityacc = sympy.Matrix([9.81, 0.0, 0.0])
    if is_anth:
        joint_offset = np.array([0.0, 0.0, np.pi/2.0, 0.0, 0.0, np.pi/2.0, 0.0])
    else:
        joint_offset =  np.array([0.0, 0.0, np.pi/2.0, 0.0, 0.0, 0.0, 0.0])

def get_actual_q(q_in: np.ndarray) -> np.ndarray:
    return q_in + joint_offset

# 构建最小参数集合
rbt = sympybotics.RobotDynCode(rbtdef, verbose=True)
rbt.calc_base_parms()
model_path = os.path.join(model_dir, 'model.pkl')
with open(model_path, 'wb') as file:
    pickle.dump(rbt, file)
with open(model_path, "rb") as file:
    rbt = pickle.load(file)

# 数据滤波, 差分和回归数据生成
fc_mult = 10
fc = fc_mult * ( wf / (2*np.pi) * L )
fc_q = fc
fc_dq = fc
fc_ddq = fc
fc_tau = float('Inf')  # fc
si = int(3/dt)
sf = -int(1/dt)
rec_h = dt # dt
def process_regression_data(traj_name):
    # 使用 Path / 拼接，并在 read_data 时转成 str
    rbtlogfile = rec_dir / f'rbtlog_{traj_name}.dat'
    trajreffile = traj_dir / f'{traj_name}.dat'
    print("rbtlogfile:", rbtlogfile)
    print("trajreffile:", trajreffile)
    t_raw, q_raw, tau_raw, t_ref, q_ref = read_data(
        dof=rbt.dof, h=rec_h,
        rbtlogfile=str(rbtlogfile),
        trajreffile=str(trajreffile)
    )
    q_raw = get_actual_q(q_raw) # 从实际的 q 得到原始的 q (对应DH参数)
    q, dq, ddq, tau = diff_and_filt_data(
        dof=rbt.dof, h=rec_h, q_raw=q_raw, tau_raw=tau_raw,
        fc_q=fc_q, fc_dq=fc_dq, fc_ddq=fc_ddq, fc_tau=fc_tau
    )
    t_raw = t_raw[si:sf]; q_raw = q_raw[si:sf]; tau_raw = tau_raw[si:sf]
    q = q[si:sf]; dq = dq[si:sf]; ddq = ddq[si:sf]; tau = tau[si:sf]
    t_raw = t_raw - t_raw[0]
    t = np.array(range(q.shape[0])) * rec_h
    W, omega, Q1, R1, rho1 = gen_regr_matrices(rbt, q, dq, ddq, tau)
    return t_raw, q_raw, tau_raw, t, q, dq, ddq, tau, W, omega, Q1, R1, rho1
for traj in ['traj1', 'traj2', 'traj3']:
    t_raw, q_raw, tau_raw, t, q, dq, ddq, tau, W, omega, Q1, R1, rho1 = process_regression_data(traj)
    np.savez_compressed(str(proc_dir / f'{traj}_raw'), t=t_raw, q=q_raw, tau=tau_raw); \
    np.savez_compressed(str(proc_dir / f'{traj}_proc'), t=t, q=q, dq=dq, ddq=ddq, tau=tau); \
    np.savez_compressed(str(proc_dir / f'{traj}_regr'), W=W, omega=omega, Q1=Q1, R1=R1, rho1=rho1)
del t_raw, q_raw, tau_raw, q, dq, ddq, tau, W, omega, Q1, R1, rho1

epsilon_safemargin = 1e-6
epsilon_sdptol = 1e-7

delta = rbt.dyn.dynparms
n_delta = rbt.dyn.n_dynparms
beta = rbt.dyn.baseparms
n_beta = rbt.dyn.n_base
beta_symbs = sympy.Matrix([sympy.Symbol('beta'+str(i+1),real=True) for i in range(n_beta)])

delta_d = (rbt.dyn.Pd.T * delta)
n_delta_d = len(delta_d)

Pb = rbt.dyn.Pb

varchange_dict = dict(zip(Pb.T * delta ,  beta_symbs - ( beta - Pb.T * delta )))

I = Identity
S = skew

D_inertia_blocks = []
for i in range(rbt.dof):
    L = rbt.rbtdef.L[i]
    l = rbt.rbtdef.l[i]
    m = rbt.rbtdef.m[i]
    Di = BlockMatrix([[L,    S(l).T],
                      [S(l), I(3)*m]])
    D_inertia_blocks.append(Di.as_explicit())

D_inertia_blocks[0]  # 展示第1个连杆的 6x6 空间惯性块

D_other_blocks = []
for i in range(rbt.dof):
    if rbt.rbtdef.driveinertiamodel == 'simplified':
        D_other_blocks.append( Matrix([rbt.rbtdef.Ia[i]]) )
    if 'viscous' in rbt.rbtdef.frictionmodel:
        D_other_blocks.append( Matrix([rbt.rbtdef.fv[i]]) )
    if 'Coulomb' in rbt.rbtdef.frictionmodel:
        D_other_blocks.append( Matrix([rbt.rbtdef.fc[i]]) )

D_blocks = D_inertia_blocks + D_other_blocks
D_blocks

# 变量空间转换 Change of Variable Space
from sympybotics.dynident.regression import mrepl
varchange_dict = dict(zip(Pb.T*delta ,  beta_symbs - ( beta - Pb.T*delta )))
DB_blocks = [mrepl(Di, varchange_dict) for Di in D_blocks]

# LMI定义 LMI Definitions
from lmi_sdp import LMI_PD, LMI
DB_LMIs = list(map(LMI_PD, DB_blocks))
DB_LMIs[0]

DB_LMIs_marg = list(map(lambda lm: LMI(lm, epsilon_safemargin*eye(lm.shape[0])) , DB_blocks))
DB_LMIs_marg[0]

def dsdp5(objf, lmis, variables):
    sdpadat = lmi_sdp.to_sdpa_sparse(objf, lmis, variables)
    sdpa_in = sdpa_dir / 'sdp.dat-s'
    dsdp_out = sdpa_dir / 'dsdp5.out'
    with open(sdpa_in, 'w') as f:
        f.write(sdpadat)

    stdout = !cd "{str(sdpa_dir)}"; dsdp5 "{sdpa_in.name}" -save "{dsdp_out.name}" -gaptol $epsilon_sdptol
    print('\n'.join(stdout))

    outfile = !cat "{str(dsdp_out)}"
    sol = [float(v) for v in outfile[0].split()]
    return np.matrix(sol).T

from lmi_sdp import init_lmi_latex_printing
import lmi_sdp

# 最小二乘回归 OLS Regression
indentification_traj = 'traj1'
regr_path = proc_dir / f"{indentification_traj}_regr.npz"
regr_mats = np.load(str(regr_path), allow_pickle=False)

W = np.matrix(regr_mats['W'])
omega = np.matrix(regr_mats['omega'])
R1 = np.matrix(regr_mats['R1'])
rho1 = np.matrix(regr_mats['rho1'])

omega.shape[0] / rbt.dof
W.shape
np.linalg.cond(W)
beta_ols = (R1.T * R1).I * R1.T * rho1
beta_ols_path = solution_dir / f"{indentification_traj}_beta_ols.dat"
np.savetxt(str(beta_ols_path), beta_ols)
rho2_norm_sqr = np.linalg.norm(omega - W * beta_ols)**2

# 基于普通最小二乘（OLS）解的基参数可行性检验 Base Parameter Feasibility Test (BPFT) of OLS Solution
dict_subs = {sym: float(val) for sym, val in zip(beta_symbs, beta_ols)}
lmis_ols_bpft = [ lmi_sdp.LMI_PD(mrepl(lmi.canonical().gts, dict_subs)) for lmi in DB_LMIs ]
lmis_ols_bpft[0]
variables_ols_bpft = list(delta_d)
objf_ols_bpft = 0

# 基参数可行性修正 Base Parameter Feasibility Correction (BPFC)
u = sympy.Symbol('u')
U_beta = BlockMatrix([[Matrix([u]),       (beta_ols - beta_symbs).T],
                 [beta_ols - beta_symbs,                 I(n_beta)]])
U_beta = U_beta.as_explicit()

lmis_ols_bpfc = [LMI(U_beta)] + DB_LMIs_marg
lmis_ols_bpfc[1]

variables_ols_bpfc = [u] + list(beta_symbs) + list(delta_d)
objf_ols_bpfc = u

sol_ols_bpfc = dsdp5(objf_ols_bpfc, lmis_ols_bpfc, variables_ols_bpfc)

u_prime = sol_ols_bpfc[0,0]
beta_prime = sol_ols_bpfc[1:1+n_beta]
delta_d_prime = sol_ols_bpfc[1+n_beta:]

print(u_prime)

#  Feasible Base Parameter Estimation with Ordinary Least Squares (FBPE-OLS)     ($\beta^\star$)
dict_subs = dict(zip(beta_symbs, beta_prime.flatten().tolist()[0]))
dict_subs.update(dict(zip(delta_d, delta_d_prime.flatten().tolist()[0])))

test = 'Solution Double Check: ok'
for i, DBi in enumerate(DB_blocks):
        m = np.matrix(mrepl(DBi, dict_subs)).astype(float)
        for v in np.linalg.eigvals( m ):
            if v <= 0.0:
                print(i, v)
                test = 'Solution Double Check: fail'
beta_prime_path = solution_dir / f"{indentification_traj}_beta_prime.dat"
np.savetxt(str(beta_prime_path), beta_prime)
print(test)

u = sympy.Symbol('u')
U_rho = BlockMatrix([[Matrix([u - rho2_norm_sqr]), (rho1 - R1*beta_symbs).T],
                     [rho1 - R1*beta_symbs,                       I(n_beta)]])
U_rho = U_rho.as_explicit()

lmis_fbpe_ols = [LMI(U_rho)] + DB_LMIs_marg

variables_fbpe_ols = [u] + list(beta_symbs) + list(delta_d)

objf_fbpe_ols = u

sol_fbpe_ols = dsdp5(objf_fbpe_ols, lmis_fbpe_ols, variables_fbpe_ols)

u_star = sol_fbpe_ols[0,0]
beta_star = np.matrix(sol_fbpe_ols[1:1+n_beta])
delta_d_star = np.matrix(sol_fbpe_ols[1+n_beta:])

u_star

dict_subs = dict(zip(beta_symbs, beta_star.flatten().tolist()[0]))
dict_subs.update(dict(zip(delta_d, delta_d_star.flatten().tolist()[0])))

test = 'ok'
for DBi in DB_blocks:
        m = np.matrix(mrepl(DBi, dict_subs)).astype(float)
        for v in np.linalg.eigvals( m ):
            if v <= 0.0:
                print(v)
                test = 'fail'
print(test)

beta_star_path = solution_dir / f"{indentification_traj}_beta_star.dat"
np.savetxt(str(beta_star_path), beta_star)

# 质心位置约束
link_cuboid_hulls = [
[[ -0.1 , +0.06 ],[ -0.08 , +0.08 ],[ -0.0 , +0.16 ]],
[[ -0.0 , +0.16 ],[ -0.08 , +0.08 ],[ -0.08 , +0.08 ]],
[[ -0.11 , +0.05 ],[ -0.06 , +0.1 ],[ 0.02 , +0.18 ]],
[[ -0.0 , +0.16 ],[ -0.08 , +0.08 ],[ -0.08 , +0.08 ]],
[[ -0.08 , +0.08 ],[ -0.1 , +0.06 ],[ -0.04 , +0.12 ]],
[[ -0.08 , +0.08 ],[ -0.08 , +0.08 ],[ -0.08 , +0.08 ]],
[[ -0.04 , +0.12 ],[ -0.04 , +0.04 ],[ -0.0 , +0.16 ]],
]

robotmaxmass = 27.0

D_add_blocks = []
for i in range(rbt.dof):
    l = rbt.rbtdef.l[i]
    m = rbt.rbtdef.m[i]
    link_cuboid_hull = link_cuboid_hulls[i]
    for j in range(3):
        D_add_blocks.append( Matrix( [[  l[j] - m*link_cuboid_hull[j][0] ]] ) )
        D_add_blocks.append( Matrix( [[ -l[j] + m*link_cuboid_hull[j][1] ]] ) )
# Variable Change and LMI Definition
D_add_blocks.append(Matrix([robotmaxmass - sum(rbt.rbtdef.m)]))
DB_add_blocks = [ mrepl(Di, varchange_dict) for Di in D_add_blocks ]
DB_add_LMIs = list(map(LMI_PD, DB_add_blocks))
DB_add_LMIs_marg = list(map(lambda lm: LMI(lm, epsilon_safemargin*eye(lm.shape[0])) , DB_add_blocks))
DB_add_LMIs_marg[0]
DB_add_LMIs_marg[-1]

# dict_subs = dict(zip(beta_symbs, beta_star))
dict_subs = {sym: float(val) for sym, val in zip(beta_symbs, beta_star)}
lmis_star_bpft_add = [ lmi_sdp.LMI_PD(mrepl(lmi.canonical().gts, dict_subs)) for lmi in DB_add_LMIs_marg ]
variables_star_bpft_add = list(delta_d)
objf_star_bpft_add = delta_d[0]

dict_subs = {sym: float(val) for sym, val in zip(beta_symbs, beta_star)}
lmis_star_bpft_add = [ lmi_sdp.LMI_PD(mrepl(lmi.canonical().gts, dict_subs)) for lmi in DB_add_LMIs_marg ]
variables_star_bpft_add = list(delta_d)
objf_star_bpft_add = delta_d[0]
sol_star_bpft_add = dsdp5(objf_star_bpft_add, lmis_star_bpft_add, variables_star_bpft_add)

lmis_fbpe_ols_add = [LMI(U_rho)] + DB_LMIs_marg + DB_add_LMIs_marg
variables_fbpe_ols_add = [u] + list(beta_symbs) + list(delta_d)
objf_fbpe_ols_add = u
sol_fbpe_ols_add = dsdp5(objf_fbpe_ols_add, lmis_fbpe_ols_add, variables_fbpe_ols_add)

u_starextra = sol_fbpe_ols_add[0,0]
beta_starextra = np.matrix(sol_fbpe_ols_add[1:1+n_beta])
delta_d_starextra = np.matrix(sol_fbpe_ols_add[1+n_beta:])

dict_subs = dict(zip(beta_symbs, beta_starextra.flatten().tolist()[0]))
dict_subs.update(dict(zip(delta_d, delta_d_starextra.flatten().tolist()[0])))

test = 'ok'
for DBi in (DB_blocks + DB_add_blocks):
        m = np.matrix(mrepl(DBi, dict_subs)).astype(float)
        for v in np.linalg.eigvals( m ):
            if v <= 0.0: test = 'fail'
print(test)

beta_starextra_path = solution_dir / f"{indentification_traj}_beta_starextra.dat"
np.savetxt(str(beta_starextra_path), beta_starextra)

# 惯性矩阵定义测试 Inertia Matrix Definiteness Tests
def test_mass_matrix_psd(beta_solution):
    K = rbt.dyn.Pb.T + rbt.dyn.Kd * rbt.dyn.Pd.T
    A = np.matrix(K).astype(float)
    b = np.matrix(beta_solution).astype(float)
    x = np.linalg.pinv(A) * b
    d_test = x

    M_func_def = sympybotics.robotcodegen.robot_code_to_func( 'python', rbt.M_code, 'M', 'M_func', rbt.rbtdef)
    exec(M_func_def, globals())
    global sin, cos
    from math import sin, cos

    ntests = 10000

    ok = 0
    nok = 0
    for i in range(ntests):
        q_test = np.random.rand(rbt.dof)
        M_out = M_func( np.array(d_test).flatten().tolist(), q_test)
        M = np.matrix(M_out).reshape((rbt.dof,rbt.dof))
        try:
            c = np.linalg.cholesky(M)
        except:
            nok += 1
            continue
        ok += 1

    print('ok     %8i  %3i%%'%(ok, 100.0*ok/ntests))
    print('not ok %8i  %3i%%'%(nok, 100.0*nok/ntests))
    if nok > 0 :
        print('\nNot PSD -> Not feasible!')
    else:
        print('\nSeems feasible.')

beta_ols = np.matrix(np.loadtxt(solution_dir / f"{indentification_traj}_beta_ols.dat")).T
test_mass_matrix_psd(beta_ols)
beta_starextra = np.matrix(np.loadtxt(solution_dir / f"{indentification_traj}_beta_starextra.dat")).T
test_mass_matrix_psd(beta_starextra)

# 回归模型校验 Regression Model Validation
from collections import OrderedDict
from support_funcs.utils import ListTable

def analyse(W, omega, R1, beta):
    from numpy import mean
    from numpy.linalg import norm

    p = dict()

    n = W.shape[0]

    omega_norm = norm(omega)
    omega_mean = mean(omega)

    p['err'] = norm(omega - W * beta)
    p['merr'] = p['err'] / n

    p['se'] = p['err']**2
    p['mse'] = p['se']/(n-W.shape[1])
    p['rmse'] = p['mse']**0.5

    C = p['mse'] * (R1.T * R1).I
    p['sd'] = np.sqrt(C.diagonal()).T
    p['sd%'] = 100. * p['sd'] / np.abs(beta)

    p['relerr'] = p['err']/ omega_norm
    p['relerr%'] = p['relerr']*100.

    p['1-r2'] = p['err']**2 / norm(omega - omega_mean)**2
    p['r2'] = 1 - p['1-r2']

    return p

trajs = OrderedDict([('id', 'traj1'),
                     ('vA', 'traj2'),
                     ('vB', 'traj3')])


betas = OrderedDict([('beta_ols', 'B^'),
                     ('beta_prime', 'B\''),
                     ('beta_star', 'B*'),
                     ('beta_starextra', 'B*e')])
solutions_beta = {
    betaname: np.matrix(
        np.loadtxt(solution_dir / f"{trajs['id']}_{betaname}.dat")
    ).T
    for betaname in betas
}

analysis = dict()

for traj in trajs:
    from numpy.linalg import cond

    print(traj)

    analysis[traj] = dict()

    regr_mats = np.load(str(proc_dir / f"{trajs[traj]}_regr.npz"))
    W = np.matrix(regr_mats['W'])
    omega = np.matrix(regr_mats['omega'])
    R1 = np.matrix(regr_mats['R1'])

    analysis[traj]['cond'] = cond(W)

    analysis[traj]['betas'] = dict()

    for betaname in betas:
        beta = solutions_beta[betaname]
        analysis[traj]['betas'][betaname] = analyse(W, omega, R1, beta)

    del W, omega, R1

prop = lambda x: x['relerr%']
form = '%.2f'

table = ListTable()
table.append([''] + ['cond'] + list(betas.values()))
for (traj, trajname) in zip(trajs, ['identification', 'validation A', 'validation B', 'validation C']):
    row = [traj] + ['%.0f'%analysis[traj]['cond']] + [ form%prop(analysis[traj]['betas'][b]) for b in betas]
    table.append(row)
prop = lambda x: x['relerr%']
form = '%.2f'

table = ListTable()
table.append([''] + ['cond'] + list(betas.values()))
for (traj, trajname) in zip(trajs, ['identification', 'validation A', 'validation B', 'validation C']):
    row = [traj] + ['%.0f'%analysis[traj]['cond']] + [ form%prop(analysis[traj]['betas'][b]) for b in betas]
    table.append(row)
import yaml
from pathlib import Path
cad_params_path = cad_dir / 'cad_params.yml'  # 使用 Path 拼接
with open(str(cad_params_path), 'r') as f:
    wam7_cad = yaml.safe_load(f)  # 使用 safe_load
delta_cad = sympy.Matrix([wam7_cad.get(str(d), d) for d in rbt.dyn.dynparms])
beta_cad = (rbt.dyn.Pb.T + rbt.dyn.Kd * rbt.dyn.Pd.T) * delta_cad

form = '%.4g'

table = ListTable()
header = ['', 'Bcad', 'B^', '% std dev B^', 'B*e']
table.append(header)
for i, b in enumerate(rbt.dyn.baseparms):
    if beta_cad[i].is_Number:
        cad_v = sympy.N(beta_cad[i], 4)
    elif beta_cad[i].is_Symbol:
        cad_v = '---'
    else:
        cad_v = sympy.N(beta_cad[i], 4)
    row = ['%.7s ...'%b if len(str(b)) > 7 else str(b), cad_v] # simplified view
    # row = [str(b), cad_v] # expanded view
    row += [form%solutions_beta['beta_ols'][i,0], '%.2g'%analysis['id']['betas']['beta_ols']['sd%'][i,0]]
    row += [form%solutions_beta['beta_starextra'][i,0]]
    table.append(row)
table

## 3 估算力矩校验可视化 Torque Plot

In [ ]:
traj = 'vA'
betaname = 'beta_starextra'
regr_mats = np.load(str(proc_dir / f"{trajs[traj]}_regr.npz"))
W = np.matrix(regr_mats['W'])
omega = np.matrix(regr_mats['omega'])

regr_mats = np.load(str(proc_dir / f"{trajs[traj]}_proc.npz"))
t = regr_mats['t']
tau_proc = np.matrix(regr_mats['tau'])

regr_mats = np.load(str(proc_dir / f"{trajs[traj]}_raw.npz"))
tau_raw = np.matrix(regr_mats['tau'])

beta_solution = np.matrix(np.loadtxt(str(solution_dir / f"{trajs['id']}_{betaname}.dat"))).T
tau = tau_raw
omega_pred = W * beta_solution
tau_pred = omega_pred.reshape( int(omega_pred.shape[0]/tau.shape[1]), tau.shape[1] )
err_pred = tau - tau_pred
joint = tau.shape[1]
total = tau.shape[0]

from matplotlib import pyplot as plt
plt.close()

plt.figure(figsize=(12,8))

axes = []
for i in range(tau.shape[1]):
    ax = plt.subplot(3, 3, i+1)
    ax.plot(t[:total], tau_raw[:,i][:total], label="Measured torque", color='tab:red', linewidth=1)
    ax.plot(t[:total], tau_pred[:,i][:total], label="Estimated torque", color='tab:blue', linewidth=1)
    # ax.plot(t[:total],err_pred[:,i][:total], label="Error")
    ax.set_title("Joint %d" % (i+1))
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Torque (Nm)")
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='upper right', fontsize='small')
    axes.append(ax)

plt.tight_layout()

plt.show()